# Phase 1: Function Calling (Tool Use)

Function calling is the bridge between LLMs (which just generate text) and Agents (which take actions). 

Instead of an LLM guessing an answer, you provide it with a list of **Tools** (Python functions). If the LLM realizes it needs real-time data or a specific calculation, it will output a request to run that tool instead of a text response.

In [ ]:
# Make sure you have the necessary packages installed
#!pip install langchain-core langchain-ollama

### 1. Define a Python Function
First, we define a normal Python function. The crucial part here is the **Docstring** and **Type Hints**. The LLM reads these to understand what the tool does and what arguments it requires.

In [1]:
from langchain_core.tools import tool

@tool
def get_stock_price(ticker: str) -> float:
    """
    Fetches the current stock price for a given company ticker symbol.
    
    Args:
        ticker: The stock ticker symbol (e.g., 'AAPL', 'GOOGL', 'MSFT').
    """
    print(f"--> [SYSTEM] Executing get_stock_price for {ticker}...")
    
    # In a real app, you would call a financial API like Yahoo Finance here.
    # For this tutorial, we will return some fake prices.
    prices = {
        "AAPL": 175.50,
        "GOOGL": 140.20,
        "MSFT": 410.00
    }
    return prices.get(ticker.upper(), 0.0)

### 2. Setup the LLM and Bind the Tool
We tell the LLM that this tool exists by 'binding' it.

In [2]:
from langchain_ollama import ChatOllama

# Note: Not all models support function calling well. 
# qwen2.5, llama3.1, or mistral are generally good at it.
llm = ChatOllama(model="qwen2.5:7b-instruct", temperature=0)

# Bind the tool to the LLM
llm_with_tools = llm.bind_tools([get_stock_price])

c:\Users\sujat\projects\AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 3. Ask a Question!
Now let's ask the LLM a question that requires the tool. Watch how the LLM responds. It won't give you a text answer; it will return a `tool_calls` object!

In [3]:
query = "Can you check the current stock price of Apple?"

print(f"User: {query}\n")
response = llm_with_tools.invoke(query)

print("LLM Response Text (Notice it's empty!):", repr(response.content))
print("\nLLM Tool Calls:")
print(response.tool_calls)

User: Can you check the current stock price of Apple?

LLM Response Text (Notice it's empty!): ''

LLM Tool Calls:
[{'name': 'get_stock_price', 'args': {'ticker': 'AAPL'}, 'id': 'a2114205-3362-4558-af9c-2e905fd39fd9', 'type': 'tool_call'}]


### 4. Execute the Tool (The Action)
The LLM told us what function it wants to run. Now, we (the system) must actually run the python code and return the result back to the LLM so it can formulate a final answer.

In [4]:
from langchain_core.messages import HumanMessage, ToolMessage

# 1. Start our conversation history
messages = [HumanMessage(content=query)]

# 2. Add the LLM's request to use the tool to the history
messages.append(response)

# 3. Check if the LLM made a tool call
if response.tool_calls:
    for tool_call in response.tool_calls:
        # Extract the name and arguments the LLM provided
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]
        
        if tool_name == "get_stock_price":
            # Execute our Python function!
            result = get_stock_price.invoke(tool_args)
            
            # Create a ToolMessage to send the result back to the LLM
            tool_msg = ToolMessage(content=str(result), tool_call_id=tool_call["id"])
            messages.append(tool_msg)

# 4. Let the LLM see the tool result and give a final answer
print("\n--- Sending tool result back to LLM... ---\n")
final_response = llm_with_tools.invoke(messages)
print("Final Answer:", final_response.content)

--> [SYSTEM] Executing get_stock_price for AAPL...

--- Sending tool result back to LLM... ---

Final Answer: The current stock price of Apple (AAPL) is $175.50.
